# Classificador de Imagens

Datasets como o MNIST e o CIFAR-10 já vêm organizados e rotulados. Na prática, o problema que queremos resolver quase nunca tem um dataset pronto: é preciso juntar as imagens, organizá-las em classes e separar o que é treino do que é avaliação.

Neste material, cada um monta o **seu próprio dataset**, baixando imagens de um buscador, e treina um classificador para ele. O caminho é:

1. baixar e limpar as imagens;
2. aumentar o dataset com **data augmentation**;
3. escolher uma rede pré-treinada e adaptar a arquitetura ao problema;
4. treinar, avaliar e testar com imagens novas.

In [ ]:
import hashlib
import os
import random
import shutil
from io import BytesIO

import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
from icrawler.builtin import BingImageCrawler
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Montando o dataset

### Baixando as imagens

Usamos a biblioteca **iCrawler** (`pip install icrawler`) para buscar um termo no Bing e salvar as imagens dos resultados em uma pasta. O SafeSearch do Bing fica no modo estrito, pelo mesmo cookie que o site usa para guardar essa preferência; ainda assim, revise as imagens.

Resultados de busca vêm com arquivos corrompidos, imagens minúsculas e repetições. A função `download_images` remove esses arquivos e continua baixando até ter `n_images` imagens. Remover repetições importa: se uma cópia cai no treino e a outra na avaliação, o modelo é avaliado em uma imagem que já viu.

In [ ]:
def clean_folder(folder, min_size=64):
    seen_hashes = set()
    for name in sorted(os.listdir(folder)):
        path = os.path.join(folder, name)

        try:
            with Image.open(path) as image:
                image.load()
                too_small = min(image.size) < min_size
        except Exception:
            too_small = True   # arquivo que não abre é removido

        with open(path, "rb") as f:
            file_hash = hashlib.md5(f.read()).hexdigest()   # arquivos idênticos têm o mesmo hash

        if too_small or file_hash in seen_hashes:
            os.remove(path)
        else:
            seen_hashes.add(file_hash)


def download_images(term, folder, n_images=50, max_rounds=10):
    os.makedirs(folder, exist_ok=True)
    offset = 0

    for _ in range(max_rounds):
        n_missing = n_images - len(os.listdir(folder))
        if n_missing <= 0:
            break

        crawler = BingImageCrawler(storage={"root_dir": folder}, log_level=50)
        # SafeSearch estrito: o padrão do Bing é o moderado, que deixa passar conteúdo adulto
        crawler.session.cookies.set("SRCHHPGUSR", "ADLT=STRICT", domain=".bing.com")
        crawler.crawl(keyword=term, offset=offset, max_num=n_missing, file_idx_offset="auto")

        clean_folder(folder)
        offset += n_images   # se faltar imagem, a próxima busca começa nos resultados seguintes

    print(f"{term}: {len(os.listdir(folder))} imagens")

### Exercício: o seu dataset

Escolha um problema de classificação e defina as classes no dicionário `search_terms`: a chave é o nome da classe (vira o nome da pasta) e o valor é o termo usado na busca. O exemplo comentado separa três salgados.

Algumas dicas para escolher:

- comece com 2 a 5 classes e cerca de 50 imagens por classe;
- termos específicos trazem menos lixo: "coxinha de frango" funciona melhor que "coxinha";
- classes parecidas entre si deixam o problema mais interessante, mas também mais difícil.

In [ ]:
dataset_dir = "data/meu_dataset"

# exemplo:
# search_terms = {
#     "coxinha": "coxinha de frango salgado",
#     "pao_de_queijo": "pão de queijo mineiro",
#     "pastel": "pastel de feira frito",
# }

search_terms = {
    # "nome_da_classe": "termo de busca",
}

for label, term in search_terms.items():
    download_images(term, os.path.join(dataset_dir, label), n_images=50)

A limpeza automática não pega imagens que simplesmente não são da classe: um pastel na busca por coxinha, um logotipo, uma foto de cardápio. Vale olhar as imagens de cada classe, apagar à mão as que estiverem erradas e rodar o download de novo para completar as pastas.

In [ ]:
fig, axes = plt.subplots(len(search_terms), 6, figsize=(14, 2.5 * len(search_terms)))
for row, label in zip(axes, search_terms):
    folder = os.path.join(dataset_dir, label)
    names = sorted(os.listdir(folder))[:6]
    for ax, name in zip(row, names):
        ax.imshow(Image.open(os.path.join(folder, name)).convert("RGB"))
        ax.set_title(f"{label}/{name}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()

### Treino, validação e teste

Dividimos as imagens de cada classe em três conjuntos:

- **treino**: usado para ajustar os pesos;
- **validação**: usado para acompanhar o treinamento e comparar escolhas (arquitetura, aumento de dados, taxa de aprendizado);
- **teste**: usado uma única vez, no final, depois de fechadas todas as escolhas.

A divisão é feita copiando os arquivos para a estrutura que o `datasets.ImageFolder` espera, uma pasta por classe:

```
data/meu_dataset_split/
    train/<classe_1>/  train/<classe_2>/  ...
    val/...
    test/...
```

In [ ]:
def split_dataset(source_dir, target_dir, val_ratio=0.15, test_ratio=0.15, seed=42):
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)   # refaz a divisão do zero

    random.seed(seed)
    for label in sorted(os.listdir(source_dir)):
        names = sorted(os.listdir(os.path.join(source_dir, label)))
        random.shuffle(names)

        n_val = int(len(names) * val_ratio)
        n_test = int(len(names) * test_ratio)
        splits = {
            "val": names[:n_val],
            "test": names[n_val:n_val + n_test],
            "train": names[n_val + n_test:],
        }

        for split, split_names in splits.items():
            os.makedirs(os.path.join(target_dir, split, label))
            for name in split_names:
                shutil.copy(os.path.join(source_dir, label, name), os.path.join(target_dir, split, label, name))

        print(f"{label}: {len(splits['train'])} treino, {len(splits['val'])} validação, {len(splits['test'])} teste")


split_dir = "data/meu_dataset_split"
split_dataset(dataset_dir, split_dir)

## Data augmentation

Com poucas dezenas de imagens por classe, o modelo vê pouca variação de cada objeto. O **data augmentation** sorteia uma transformação nova a cada vez que uma imagem de treino é carregada, então em cada época o modelo vê uma versão diferente dela, com o mesmo rótulo.

Abaixo, o efeito de algumas transformações do `torchvision.transforms` em uma imagem do treino.

In [ ]:
first_class = sorted(search_terms)[0]
first_folder = os.path.join(split_dir, "train", first_class)
sample_image = Image.open(os.path.join(first_folder, sorted(os.listdir(first_folder))[0])).convert("RGB")
sample_image = transforms.Resize(256)(sample_image)

plt.imshow(sample_image)
plt.axis("off")
plt.show()

- `RandomHorizontalFlip`: espelha a imagem. Só vale se o objeto espelhado continua sendo da mesma classe.
- `RandomRotation`: gira por um ângulo sorteado.
- `RandomResizedCrop`: recorta uma região aleatória e redimensiona, simulando objetos mais perto, mais longe ou deslocados.
- `ColorJitter`: altera brilho, contraste, saturação e matiz, simulando iluminação e câmeras diferentes.
- `RandomGrayscale`: tira a cor, para o modelo não depender só dela.
- `RandomPerspective`: distorce como se a foto fosse tirada de outro ângulo.

In [ ]:
augmentations = {
    "HorizontalFlip": transforms.RandomHorizontalFlip(p=0.5),
    "Rotation(30)": transforms.RandomRotation(30),
    "ResizedCrop": transforms.RandomResizedCrop(224, scale=(0.3, 1.0)),
    "ColorJitter": transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
    "Grayscale": transforms.RandomGrayscale(p=0.5),
    "Perspective": transforms.RandomPerspective(distortion_scale=0.5, p=1.0),
}

torch.manual_seed(0)
fig, axes = plt.subplots(len(augmentations), 5, figsize=(12, 2.6 * len(augmentations)))
for row, (name, augmentation) in zip(axes, augmentations.items()):
    for ax in row:
        ax.imshow(augmentation(sample_image))
        ax.axis("off")
    row[0].set_title(name, loc="left")
plt.tight_layout()
plt.show()

### Exercício: o aumento de dados do seu problema

Monte o `train_transform` com as transformações que fazem sentido para as suas classes: a imagem transformada precisa continuar realista e da mesma classe. Espelhar não muda um salgado, mas muda uma placa com texto; tirar a cor atrapalha se é a cor que separa as classes. Uma alternativa é a política pronta `transforms.TrivialAugmentWide()`.

O pipeline termina em $224 \times 224$, tensor e normalização do ImageNet. Validação e teste não recebem aumento, só redimensionamento e recorte central.

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    # acrescente as transformações aleatórias escolhidas para o seu dataset
    transforms.Resize(256),       # pode ser trocado por um RandomResizedCrop(224, ...)
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

*Explique aqui as suas escolhas: quais transformações usou, quais descartou e por quê.*

### Datasets e DataLoaders

Cada conjunto recebe o seu `transform`. As imagens do lote de treino mostradas abaixo já passaram pelo `train_transform`: confira se elas continuam reconhecíveis.

In [ ]:
train_set = datasets.ImageFolder(os.path.join(split_dir, "train"), transform=train_transform)
val_set = datasets.ImageFolder(os.path.join(split_dir, "val"), transform=eval_transform)
test_set = datasets.ImageFolder(os.path.join(split_dir, "test"), transform=eval_transform)

train_dataloader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_dataloader = DataLoader(val_set, batch_size=64, num_workers=2)
test_dataloader = DataLoader(test_set, batch_size=64, num_workers=2)

classes = train_set.classes
num_classes = len(classes)
print(classes)
print(f"{len(train_set)} treino, {len(val_set)} validação, {len(test_set)} teste")

In [ ]:
def denormalize(image):
    image = image.permute(1, 2, 0)               # [3, H, W] -> [H, W, 3]
    image = image * torch.tensor(std) + torch.tensor(mean)
    return image.clamp(0, 1)


images, labels = next(iter(train_dataloader))
print(images.shape)   # [32, 3, 224, 224]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, label in zip(axes.flat, images, labels):
    ax.imshow(denormalize(image))
    ax.set_title(classes[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

## Escolhendo o modelo pré-treinado

O `torchvision.models` tem dezenas de arquiteturas treinadas no ImageNet. Elas diferem principalmente no compromisso entre acurácia, tamanho e velocidade:

- **ResNet** (18, 34, 50, ...): conexões residuais, que somam a entrada de um bloco à sua saída e permitem treinar redes profundas. A ResNet-18 é pequena e rápida; a ResNet-50 é mais precisa e bem maior.
- **MobileNet V3**: feita para celulares. Usa convoluções separáveis em profundidade, muito mais baratas que as convoluções comuns. Poucos parâmetros e rápida até em CPU.
- **EfficientNet**: escala largura, profundidade e resolução da rede de forma equilibrada. A B0 tem ótima acurácia para o seu tamanho.
- **ConvNeXt**: uma CNN modernizada com escolhas de projeto vindas dos Transformers. Mais precisa, mas mais pesada.
- **ViT** (*Vision Transformer*): divide a imagem em blocos de $16 \times 16$ e processa como uma sequência, com Transformers. Precisa de mais dados para o ajuste fino.

O objeto de pesos de cada arquitetura guarda informações sobre ela, como o número de parâmetros e a acurácia no ImageNet. Ler essas informações não baixa os pesos.

In [ ]:
candidates = {
    "resnet18": models.ResNet18_Weights.DEFAULT,
    "resnet50": models.ResNet50_Weights.DEFAULT,
    "mobilenet_v3_small": models.MobileNet_V3_Small_Weights.DEFAULT,
    "mobilenet_v3_large": models.MobileNet_V3_Large_Weights.DEFAULT,
    "efficientnet_b0": models.EfficientNet_B0_Weights.DEFAULT,
    "convnext_tiny": models.ConvNeXt_Tiny_Weights.DEFAULT,
    "vit_b_16": models.ViT_B_16_Weights.DEFAULT,
}

print(f"{'modelo':20s} {'parâmetros':>12s} {'acurácia ImageNet':>18s}")
for name, weights in candidates.items():
    n_params = weights.meta["num_params"]
    acc = weights.meta["_metrics"]["ImageNet-1K"]["acc@1"]
    print(f"{name:20s} {n_params / 1e6:10.1f} M {acc:17.1f}%")

### Adaptando a arquitetura

Qualquer que seja a escolha, a adaptação segue três passos:

1. carregar a rede com os pesos pré-treinados;
2. congelar os parâmetros com `requires_grad_(False)` (todos, ou só as primeiras camadas, se quiser fazer ajuste fino);
3. substituir a última camada, que tem 1000 saídas, por uma com `num_classes` saídas.

A diferença entre as arquiteturas está no passo 3: a última camada tem um nome diferente em cada uma. Na ResNet é `model.fc`; em outras, fica dentro de um `nn.Sequential` chamado `classifier` ou `heads`. Para descobrir, imprima o modelo e procure a última `nn.Linear`. Por exemplo, a MobileNet V3 Small:

In [ ]:
example = models.mobilenet_v3_small()   # sem pesos, só para ver a arquitetura
print(example.classifier)

Aqui a última camada é `example.classifier[3]`, com `in_features=1024`. Trocar só ela mantém o resto do classificador (a camada de 1024, a ativação e o dropout), que também é pré-treinado.

Também é possível trocar a camada final por uma pequena rede, por exemplo `nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))`. O dropout ajuda quando o dataset é pequeno.

### Exercício: o seu modelo

Escolha uma das arquiteturas (de preferência diferente da ResNet-18) e adapte-a ao seu dataset. A célula seguinte confere se a saída tem o formato certo.

In [ ]:
# weights = models.<Arquitetura>_Weights.DEFAULT
# model = models.<arquitetura>(weights=weights)
# print(model)

# congele os parâmetros e troque a última camada

# model = model.to(device)

In [ ]:
images, labels = next(iter(val_dataloader))
model.eval()
with torch.no_grad():
    logits = model(images.to(device))

assert logits.shape == (len(images), num_classes), f"saída {tuple(logits.shape)}, esperado ({len(images)}, {num_classes})"

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{n_total:,} parâmetros, {n_trainable:,} treináveis")

## Treinamento

A cada época, guardamos a perda e a acurácia de treino e de validação, para desenhar as curvas e perceber sobreajuste.

A função `evaluate` calcula a perda média e a acurácia do modelo em um `DataLoader`, sem atualizar os pesos. Ela é usada na validação durante o treino e, no final, no teste.

In [ ]:
criterion = nn.CrossEntropyLoss()


def evaluate(model, dataloader):
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item() * len(images)
            correct += (logits.argmax(dim=1) == labels).sum().item()
    n = len(dataloader.dataset)
    return total_loss / n, correct / n

### Exercício: o laço de treinamento

Complete a função `train`. Ela deve:

- criar um otimizador Adam só com os parâmetros treináveis (`requires_grad=True`);
- a cada época, colocar o modelo em modo de treino (`model.train()`) e percorrer o `train_dataloader`, calculando a perda, os gradientes e atualizando os pesos;
- acumular a perda e os acertos de treino ao longo da época, ponderando a perda pelo tamanho de cada lote (o último lote pode ser menor);
- ao fim de cada época, calcular a perda e a acurácia de validação com `evaluate` e guardar os quatro valores no `history`;
- devolver o `history`, que é usado para desenhar as curvas.

Se você descongelou parte da rede para fazer ajuste fino, use uma taxa de aprendizado menor (como `1e-4`), para não destruir os pesos pré-treinados nas primeiras épocas.

In [ ]:
def train(model, train_dataloader, val_dataloader, num_epochs, lr=1e-3):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    # crie o otimizador e escreva o laço de épocas

    return history

In [ ]:
num_epochs = 10
lr = 1e-3

torch.manual_seed(42)
history = train(model, train_dataloader, val_dataloader, num_epochs, lr)

In [ ]:
assert all(len(values) == num_epochs for values in history.values()), "o history deve ter um valor por época em cada chave"

Nas curvas, a acurácia de treino pode ficar **abaixo** da de validação nas primeiras épocas. Isso não é erro: as imagens de treino passam pelo aumento de dados e são mais difíceis que as de validação, e a perda de treino é a média ao longo da época, enquanto o modelo ainda estava aprendendo. Se, ao contrário, o treino continuar subindo enquanto a validação cai, o modelo está sobreajustando.

In [ ]:
epochs = range(1, num_epochs + 1)

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
ax_loss.plot(epochs, history["train_loss"], label="treino")
ax_loss.plot(epochs, history["val_loss"], label="validação")
ax_loss.set_xlabel("época")
ax_loss.set_ylabel("perda")
ax_loss.legend()

ax_acc.plot(epochs, history["train_acc"], label="treino")
ax_acc.plot(epochs, history["val_acc"], label="validação")
ax_acc.set_xlabel("época")
ax_acc.set_ylabel("acurácia")
ax_acc.legend()
plt.show()

## Avaliação no teste

Com o modelo escolhido e treinado, medimos o desempenho no conjunto de teste, que não participou de nenhuma decisão até aqui. A **matriz de confusão** mostra, para cada classe real (linhas), quantas imagens foram previstas como cada classe (colunas): os erros ficam fora da diagonal.

In [ ]:
test_loss, test_acc = evaluate(model, test_dataloader)
print(f"acurácia de teste {test_acc:.3f}")

all_labels = []
all_predictions = []
model.eval()
with torch.no_grad():
    for images, labels in test_dataloader:
        predictions = model(images.to(device)).argmax(dim=1).cpu()
        all_labels.extend(labels.tolist())
        all_predictions.extend(predictions.tolist())

ConfusionMatrixDisplay.from_predictions(all_labels, all_predictions, display_labels=classes, cmap="Blues")
plt.show()

In [ ]:
images, labels = next(iter(test_dataloader))
with torch.no_grad():
    predictions = model(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, i in zip(axes.flat, torch.randperm(len(images))):
    ax.imshow(denormalize(images[i]))
    color = "green" if predictions[i] == labels[i] else "red"
    ax.set_title(f"previsto: {classes[predictions[i]]}\nreal: {classes[labels[i]]}", color=color)
    ax.axis("off")
plt.tight_layout()
plt.show()

Salvamos os pesos para usar o modelo depois sem treinar de novo.

In [ ]:
torch.save(model.state_dict(), "meu_modelo.pth")

## Testando com imagens novas

As imagens de teste vieram da mesma busca que as de treino, então se parecem com elas: mesmo estilo de foto, mesmos sites. Um teste mais honesto usa imagens de outra fonte: fotos tiradas por você, imagens de outro buscador ou de outra busca.

A função abaixo carrega uma imagem de um arquivo ou de uma URL, aplica o `eval_transform` e mostra a imagem com a probabilidade de cada classe.

In [ ]:
def load_image(source):
    if source.startswith("http"):
        response = requests.get(source, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        return Image.open(BytesIO(response.content)).convert("RGB")
    return Image.open(source).convert("RGB")


def predict(model, image):
    x = eval_transform(image).unsqueeze(0).to(device)   # [1, 3, 224, 224]
    model.eval()
    with torch.no_grad():
        logits = model(x)                               # [1, num_classes]
    return logits[0].softmax(dim=0).cpu()


def show_prediction(model, source):
    image = load_image(source)
    probabilities = predict(model, image)

    fig, (ax_image, ax_bar) = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={"width_ratios": [1, 1.4]})
    ax_image.imshow(image)
    ax_image.set_title(classes[probabilities.argmax()])
    ax_image.axis("off")
    ax_bar.barh(classes, probabilities.numpy())
    ax_bar.set_xlim(0, 1)
    ax_bar.set_xlabel("probabilidade")
    plt.tight_layout()
    plt.show()

Coloque fotos suas em uma pasta e passe o caminho de cada uma para `show_prediction`, ou passe a URL de uma imagem da internet.

In [ ]:
# show_prediction(model, "minha_foto.jpg")
# show_prediction(model, "https://...")

Repare também no que acontece com uma imagem que não é de nenhuma das classes: o modelo sempre escolhe uma delas, muitas vezes com alta confiança. A softmax distribui a probabilidade entre as classes conhecidas e não tem como dizer "nenhuma delas".